# 规则置换风险分析样例

演示如何使用hscredit的swapin_risk_prediction功能进行规则置换风险预估。
适用于金融信贷业务中规则置换时分析置入置出的风险指标。

In [1]:
import sys
sys.path.insert(0, '/Users/xiaoxi/CodeBuddy/hscredit/hscredit')

import pandas as pd
import numpy as np
from hscredit import Rule
from hscredit.report import swapin_risk_prediction, SwapinRiskAnalyzer

## 加载示例数据

In [2]:
# 加载示例数据
df = pd.read_excel('/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/hscredit.xlsx')

print(f"数据形状: {df.shape}")
print(f"列名: {df.columns.tolist()}")

# 创建目标变量
df['target'] = (df['MOB1'] > 15).astype(int)
print(f"\n目标变量分布:")
print(df['target'].value_counts())

数据形状: (22729, 12)
列名: ['MOB1', 'MOB2', '青云24', '游昆定制分80', '百融定制分V9', '中智小牛分C3', '度小满欺诈因子V6PRO多头版', '百行百川分FPV1', '设备黑名单', '放款期数']

目标变量分布:
target
0    21122
1     1607
Name: count, dtype: int64


## 示例1: 基础置换分析

场景：评估将"青云24 > 600"置换为"青云24 > 700"的效果

In [3]:
print("=" * 80)
print("示例1: 基础置换分析")
print("=" * 80)

# 原规则
base_rule = Rule('青云24 > 600')
# 新规则
new_rule = Rule('青云24 > 700')

print("\n场景：将 '青云24 > 600' 置换为 '青云24 > 700'")
print("-" * 60)

# 分析置换效果
result_df = swapin_risk_prediction(
    base=df,
    test=df,
    swapin_rules=new_rule,
    feature='青云24',
    target='target',
    method='quantile',
    max_n_bins=5
)

print("\n置换风险分析结果:")
result_df

示例1: 基础置换分析

场景：将 '青云24 > 600' 置换为 '青云24 > 700'
------------------------------------------------------------

置换风险分析结果:


,样本总数,坏样本数,坏样本率,样本占比,LIFT值,坏账改善,风险拒绝比
规则详情,,,,,,,
青云24 > 700,1550.0,77.58680,0.050056,0.068195,0.707978,0.292022,4.282180
剩余样本,21179.0,1529.41731,0.072214,0.931805,1.021372,-0.021372,-0.022936
合计,22729.0,1607.00411,0.070703,1.000000,1.000000,0.000000,0.000000


## 示例2: 使用prior_rules进行分层分析

场景：已有规则"百融定制分V9 > 500"，想评估换成"百融定制分V9 > 650"的效果

In [4]:
print("=" * 80)
print("示例2: 使用prior_rules进行分层分析")
print("=" * 80)

# 原规则（先验规则）
old_rule = Rule('百融定制分V9 > 500')
# 新规则（更严格）
strict_rule = Rule('百融定制分V9 > 650')

print("\n场景：将 '百融定制分V9 > 500' 置换为 '百融定制分V9 > 650'")
print("-" * 60)

# 使用report方法的prior_rules参数进行分析
report = strict_rule.report(
    df,
    target='target',
    desc='百融定制分>650',
    prior_rules=old_rule,
    filter_cols=['规则分类', '分箱', '样本总数', '样本占比', '坏样本率', '坏账改善', '风险拒绝比']
)

print("\n分层分析报告:")
report

示例2: 使用prior_rules进行分层分析

场景：将 '百融定制分V9 > 500' 置换为 '百融定制分V9 > 650'
------------------------------------------------------------

分层分析报告:


,规则分类,指标名称,指标含义,分箱,样本总数,样本占比,坏样本率,坏账改善,风险拒绝比
0,先验规则,百融定制分V9 > 500,百融定制分>650,命中,22682,0.997932,0.070585,-0.001666,-0.001669
1,先验规则,百融定制分V9 > 500,百融定制分>650,未命中,47,0.002068,0.127660,0.001666,0.805585
2,验证规则,百融定制分V9 > 650,百融定制分>650,命中,0,0.000000,0.000000,0.000000,NaN
3,验证规则,百融定制分V9 > 650,百融定制分>650,未命中,47,1.000000,0.127660,0.000000,0.000000


In [5]:
# 提取关键指标进行分析
if '先验规则' in report['规则分类'].values:
    prior_hit = report[report['规则分类'] == '先验规则']['样本总数'].sum()
    print(f"\n先验规则命中样本数: {prior_hit}")
    
if '验证规则' in report['规则分类'].values:
    new_hit = report[(report['规则分类'] == '验证规则') & (report['分箱'] == '命中')]['样本总数'].sum()
    print(f"新规则在剩余样本中命中: {new_hit}")


先验规则命中样本数: 22729
新规则在剩余样本中命中: 0


## 示例3: 多规则组合置换分析

场景：评估增加"设备黑名单"规则的效果（在已有规则基础上）

In [6]:
print("=" * 80)
print("示例3: 多规则组合置换分析")
print("=" * 80)

# 已有规则组合
existing_rules = Rule('青云24 > 600') | Rule('百融定制分V9 > 500')
# 新增规则
new_added_rule = Rule('设备黑名单 > 0')

print("\n场景：在已有规则(青云24>600 或 百融>500)基础上增加设备黑名单规则")
print("-" * 60)

report_addition = new_added_rule.report(
    df,
    target='target',
    desc='设备黑名单规则',
    prior_rules=existing_rules,
    filter_cols=['规则分类', '分箱', '样本总数', '样本占比', '坏样本率']
)

print("\n新增规则效果分析:")
report_addition

示例3: 多规则组合置换分析

场景：在已有规则(青云24>600 或 百融>500)基础上增加设备黑名单规则
------------------------------------------------------------

新增规则效果分析:


,规则分类,指标名称,指标含义,分箱,样本总数,样本占比,坏样本率
0,先验规则,青云24 > 600 | 百融定制分V9 > 500,设备黑名单规则,命中,22690,0.998284,0.070692
1,先验规则,青云24 > 600 | 百融定制分V9 > 500,设备黑名单规则,未命中,39,0.001716,0.076923
2,验证规则,设备黑名单 > 0,设备黑名单规则,命中,25,0.641026,0.040000
3,验证规则,设备黑名单 > 0,设备黑名单规则,未命中,14,0.358974,0.142857


## 示例4: 使用SwapinRiskAnalyzer进行批量分析

使用SwapinRiskAnalyzer类提供更灵活的置换分析方式

In [7]:
print("=" * 80)
print("示例4: 使用SwapinRiskAnalyzer进行批量分析")
print("=" * 80)

# 创建分析器
analyzer = SwapinRiskAnalyzer(method='quantile', max_n_bins=5)

# 定义要分析的置换规则
test_rule = Rule('青云24 > 700')

print("\n使用SwapinRiskAnalyzer进行分析:")
print("-" * 60)

# 使用分析器进行分析
result = analyzer.analyze(
    base=df,
    test=df,
    swapin_rules=test_rule,
    feature='青云24',
    target='target'
)

print("\n置换风险分析结果:")
result

示例4: 使用SwapinRiskAnalyzer进行批量分析

使用SwapinRiskAnalyzer进行分析:
------------------------------------------------------------

置换风险分析结果:


,样本总数,坏样本数,坏样本率,样本占比,LIFT值,坏账改善,风险拒绝比
规则详情,,,,,,,
青云24 > 700,1550.0,77.58680,0.050056,0.068195,0.707978,0.292022,4.282180
剩余样本,21179.0,1529.41731,0.072214,0.931805,1.021372,-0.021372,-0.022936
合计,22729.0,1607.00411,0.070703,1.000000,1.000000,0.000000,0.000000


## 示例5: 置换决策建议

根据置换前后的风险指标，自动给出置换决策建议

In [8]:
print("=" * 80)
print("示例5: 置换决策建议")
print("=" * 80)

print("\n场景：评估将'百融>500'置换为'百融>650'的收益")
print("-" * 60)

# 原规则效果
original_rule = Rule('百融定制分V9 > 500')
original_report = original_rule.report(df, target='target', desc='原规则')

# 提取原规则指标
orig_hit_row = original_report[original_report['分箱'] == '命中']
if not orig_hit_row.empty:
    original_hit = orig_hit_row['样本总数'].values[0]
    original_bad_rate = orig_hit_row['坏样本率'].values[0]
    
    print(f"\n原规则(百融>500):")
    print(f"  拦截样本数: {original_hit}")
    print(f"  拦截样本坏样本率: {original_bad_rate:.4f}")

    # 新规则效果（在先验规则基础上）
    new_rule = Rule('百融定制分V9 > 650')
    swap_report = new_rule.report(df, target='target', desc='新规则', prior_rules=original_rule)
    
    # 计算置换指标
    prior_hit = swap_report[swap_report['规则分类'] == '先验规则']['样本总数'].sum()
    new_hit_row = swap_report[(swap_report['规则分类'] == '验证规则') & (swap_report['分箱'] == '命中')]
    
    if not new_hit_row.empty:
        new_hit = new_hit_row['样本总数'].values[0]
        new_bad_rate = new_hit_row['坏样本率'].values[0]
        
        print(f"\n置换后:")
        print(f"  先验规则命中: {prior_hit}")
        print(f"  新增命中: {new_hit}")
        print(f"  新拦截样本坏样本率: {new_bad_rate:.4f}")
        
        # 决策建议
        print(f"\n置换决策建议:")
        if new_bad_rate > original_bad_rate * 1.5:
            print("  ✅ 强烈建议置换")
            print(f"     新规则拦截样本的坏样本率({new_bad_rate:.4f})显著高于原规则({original_bad_rate:.4f})")
        elif new_bad_rate > original_bad_rate:
            print("  ⚠️  建议置换")
            print(f"     新规则拦截样本的坏样本率({new_bad_rate:.4f})高于原规则({original_bad_rate:.4f})")
        else:
            print("  ❌ 不建议置换")
            print(f"     新规则拦截样本的坏样本率({new_bad_rate:.4f})未超过原规则({original_bad_rate:.4f})")

示例5: 置换决策建议

场景：评估将'百融>500'置换为'百融>650'的收益
------------------------------------------------------------

原规则(百融>500):
  拦截样本数: 22682
  拦截样本坏样本率: 0.0706

置换后:
  先验规则命中: 22729
  新增命中: 0
  新拦截样本坏样本率: 0.0000

置换决策建议:
  ❌ 不建议置换
     新规则拦截样本的坏样本率(0.0000)未超过原规则(0.0706)


## 示例6: 结合金额加权的置换分析

考虑放款金额权重的置换分析，对于评估资金风险更有意义

In [9]:
print("=" * 80)
print("示例6: 结合金额加权的置换分析")
print("=" * 80)

# 假设有放款金额字段，我们进行金额加权的分析
# 这里使用一个模拟的放款金额字段
df['放款金额'] = np.random.uniform(1000, 10000, len(df))

print("\n场景：考虑放款金额权重的置换分析")
print("-" * 60)

# 金额加权的置换分析
result_with_amount = swapin_risk_prediction(
    base=df,
    test=df,
    swapin_rules=Rule('青云24 > 700'),
    feature='青云24',
    target='target',
    amount='放款金额',
    method='quantile',
    max_n_bins=3
)

print("\n金额加权置换分析结果:")
result_with_amount

示例6: 结合金额加权的置换分析

场景：考虑放款金额权重的置换分析
------------------------------------------------------------

金额加权置换分析结果:


(               样本总数        坏样本数      坏样本率      样本占比     LIFT值      坏账改善  \
 规则详情                                                                      
 青云24 > 700   1550.0    77.58680  0.050056  0.068195  0.707978  0.292022   
 剩余样本        21179.0  1529.41731  0.072214  0.931805  1.021372 -0.021372   
 合计          22729.0  1607.00411  0.070703  1.000000  1.000000  0.000000   
 
                风险拒绝比  
 规则详情                  
 青云24 > 700  4.282180  
 剩余样本       -0.022936  
 合计          0.000000  ,
                     样本总数          坏样本数      坏样本率      样本占比     LIFT值  \
 规则详情                                                                   
 青云24 > 700  8.592879e+06  4.301252e+05  0.050056  0.068815  0.707940   
 剩余样本        1.162770e+08  8.398997e+06  0.072233  0.931185  1.021583   
 合计          1.248698e+08  8.829122e+06  0.070707  1.000000  1.000000   
 
                 坏账改善     风险拒绝比  
 规则详情                            
 青云24 > 700  0.292060  4.244158  
 剩余样本       -0.021583 -0.023

## 总结

规则置换分析是信贷风控中常见的业务场景，主要有以下几种分析方法：

1. **swapin_risk_prediction函数**: 用于全量样本的置换风险预测
2. **prior_rules参数**: 用于分层分析，在先验规则剩余样本上评估新规则
3. **SwapinRiskAnalyzer类**: 提供更灵活的批量分析能力
4. **amount参数**: 结合放款金额权重进行风险分析

这些工具可以帮助风控人员：
- 评估规则置换的收益
- 量化新规则的风险拦截能力
- 支持规则迭代的决策制定